# FIN510 Coursework 2 Option C | Volatility forecasting

A risk team holds SPY, an ETF that tracks the S&P 500. It uses recent returns to forecast variance for the next trading day. Would a GARCH forecast improve on its existing 30-day rule enough to justify a trial? Compare both forecasts on the same trading days and make a recommendation to the team.

Use daily SPY closing prices from the course Bloomberg extract, January 2018 to December 2024. Fit GARCH(1,1) once using returns through December 2021. Keep those coefficients fixed and update both variance forecasts after each trading close. Score the forecasts from January 2022 to December 2024, then compare 2022 with 2023–2024. Keep these dates fixed for the main comparison.

Before running the results, write down what pattern across the two periods would make you consider a trial. Run the notebook in order. In Colab, upload the `.ipynb` file and run all cells. The first code cell downloads the pinned course data as needed.

## Which return belongs in Tuesday's forecast?

Suppose Monday's variance forecast was $1.00\,\%^{2}$ and Monday closes with a return of $-2.00\%$. At Monday's close you can use the squared return of $4.00\,\%^{2}$. With illustrative GARCH coefficients $\omega=0.05$, $\alpha=0.10$ and $\beta=0.85$, the forecast for Tuesday is

$$
0.05 + 0.10(-2.00)^2 + 0.85(1.00) = 1.30\,\%^{2}.
$$

The 30-day rule averages Monday's squared return and the 29 previous trading-day squared returns. Both forecasts belong on the **Tuesday** row. At Tuesday's close you can score them using Tuesday's squared return, then update both forecasts for the following trading day.

## Prepare daily SPY returns

The [course Bloomberg extract](https://github.com/quinfer/fin510-colab-notebooks/blob/3f24c99391fd874bd16c53e9a6337b6ed360bc41/resources/bloomberg_database.parquet) records closing prices in `PX_LAST`. Take the percentage change between consecutive observed closes. The first price establishes the base for the first return. Use the reported dates and observation counts to describe your sample.

In [ ]:
import hashlib
from io import BytesIO
from pathlib import Path
import subprocess
import sys
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from arch import arch_model
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "arch==8.0.0"])
    from arch import arch_model

data_url = (
    "https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/"
    "3f24c99391fd874bd16c53e9a6337b6ed360bc41/resources/"
    "bloomberg_database.parquet"
)
expected_sha256 = "9af9edd70dfe7d50aa5018bea58f3a1a80c621567014b7d5f8a49fcd3ac4db5c"
local_data = Path("bloomberg_database.parquet")

if local_data.is_file():
    payload = local_data.read_bytes()
else:
    with urlopen(data_url, timeout=30) as response:
        payload = response.read()

if hashlib.sha256(payload).hexdigest() != expected_sha256:
    raise ValueError("Use the pinned Bloomberg extract linked above.")

source = pd.read_parquet(BytesIO(payload), columns=["ticker", "date", "PX_LAST"])
spy = source.loc[source["ticker"].eq("SPY"), ["date", "PX_LAST"]].copy()
spy["date"] = pd.to_datetime(spy["date"])
prices = spy.set_index("date")["PX_LAST"].sort_index().loc["2018-01-03":"2024-12-31"]

if prices.empty or not prices.index.is_unique or not prices.gt(0).all():
    raise ValueError("Check the SPY dates and positive closing prices.")

returns = prices.pct_change(fill_method=None).dropna().mul(100)
train = returns.loc[:"2021-12-31"]
test = returns.loc["2022-01-01":"2024-12-31"]

for label, sample in [("Training", train), ("Test", test)]:
    print(f"{label} returns, {sample.index[0]:%d %b %Y} to "
          f"{sample.index[-1]:%d %b %Y} ({len(sample):,} observations)")

## Fit GARCH and issue dated forecasts

Write $r_t$ for the trading-day return in per cent and $h_t$ for the variance forecast for that day. We set the daily mean to zero for both methods. Under this assumption $r_t^2$ is a proxy for variance, in squared percentage units. It is one noisy observation for each day.

The first GARCH forecast comes from the end of the fitted training sample. For each test date, save the forecast first, then use that day's observed return to calculate the next forecast. The fitted coefficients stay fixed. For the 30-day forecast, a shift by one row moves each average to the date it forecasts.

In [ ]:
specification = arch_model(
    train, mean="Zero", vol="GARCH", p=1, q=1, dist="normal", rescale=False
)
fit = specification.fit(disp="off")
if fit.convergence_flag != 0:
    raise RuntimeError("The training fit did not converge. Inspect the model fit.")

omega = fit.params["omega"]
alpha = fit.params["alpha[1]"]
beta = fit.params["beta[1]"]
display(fit.params.to_frame("Training estimate").round(4))

next_variance = float(fit.forecast(horizon=1, reindex=False).variance.iloc[-1, 0])
garch_forecasts = []
for observed_return in test:
    garch_forecasts.append(next_variance)
    next_variance = omega + alpha * observed_return**2 + beta * next_variance

squared_return = returns.pow(2)
trailing_forecast = squared_return.rolling(30, min_periods=30).mean().shift(1)
comparison = pd.DataFrame({
    "Squared return": test.pow(2),
    "30-day forecast": trailing_forecast.reindex(test.index),
    "GARCH forecast": pd.Series(garch_forecasts, index=test.index),
})

forecast_columns = ["30-day forecast", "GARCH forecast"]
if not np.isfinite(comparison.to_numpy()).all():
    raise ValueError("Every test return needs both forecasts on the same date.")
if not comparison[forecast_columns].gt(0).all().all():
    raise ValueError("Both methods must forecast positive variance.")

display(comparison.head(3).round(4))

## Compare forecast errors

Subtract the observed squared return from each forecast on the same row. Square those errors and take their mean. This is mean squared error (MSE). Lower MSE means the forecast was closer to the squared returns under this loss rule. Variance is in $\%^{2}$, so MSE is in $\%^{4}$.

Compare the full test period with the two pre-specified periods, 2022 and 2023–2024. A few large daily returns can have a large effect on MSE. Use the plot to inspect when large forecast errors could have occurred, then check them against the dated results. The two panels share a date axis and use different vertical scales.

In [ ]:
test_periods = {
    "2022–2024": comparison,
    "2022": comparison.loc["2022"],
    "2023–2024": comparison.loc["2023":"2024"],
}

error_rows = []
for label, daily in test_periods.items():
    errors = daily[forecast_columns].sub(daily["Squared return"], axis="index")
    mse = errors.pow(2).mean()
    error_rows.append({
        "Period": label,
        "Trading days": len(daily),
        "30-day MSE": mse["30-day forecast"],
        "GARCH MSE": mse["GARCH forecast"],
    })

comparison_table = pd.DataFrame(error_rows).set_index("Period")
display(comparison_table.round(4))

In [ ]:
fig, (ax_forecast, ax_observed) = plt.subplots(
    2, 1, figsize=(10, 6), sharex=True, layout="constrained"
)
for column in forecast_columns:
    ax_forecast.plot(comparison.index, comparison[column], label=column, linewidth=1.2)
ax_forecast.set_ylabel("Forecast variance (%²)")
ax_forecast.legend(loc="upper right")
ax_forecast.set_title("SPY variance forecasts and squared returns")

ax_observed.plot(comparison.index, comparison["Squared return"], color="0.35", linewidth=0.8)
ax_observed.set_ylabel("Squared return (%²)")
ax_observed.set_xlabel("Target trading date")
ax_observed.set_xlim(comparison.index[0], comparison.index[-1])
plt.show()

## Make a risk decision

- Trace the first test forecast for each method. What was the last return available when you issued it? What return did you use to score it?
- Compare MSE on the full test and the two subperiods. Describe the size and stability of the difference. Pick a forecast date from the plot and use the dated values to explain how the two methods reacted to earlier market movements.
- Would you keep using the 30-day rule or trial GARCH alongside it for daily risk monitoring? Explain what the forecast evidence supports, what additional implementation work a trial would need, and what result could change your recommendation. Consider how using squared daily returns affects the evidence.

Use the coursework brief for submission requirements. Cite the data and explain the sample, forecast timing, model choices and limitations in your own words.